In [ ]:
# Notebook: daniel_testing.ipynb
# Fixed version of annabelle_update.ipynb
#
# Key fixes applied:
#  1. Single pip install, single SparkSession, helper functions defined once
#  2. All ML + MLflow imports consolidated in one block
#  3. RandomForestClassifier imported at the top (not buried mid-notebook)
#  4. Phases 4 & 5 read from saved Silver parquet (no ETL re-run)
#  5. Label threshold computed on training data only (prevents label leakage)
#  6. target_audience added to categorical features
#  7. toPandas() uses .sample() to avoid collecting full dataset to driver
#  8. Redundant .count() actions removed; total row count stored as variable
#  9. Phase 5 re-uses Phase 4 models for MLflow logging (no re-training)
# 10. Feature names extracted robustly from best model's own prediction df
# 11. AUC diagnostic warning added when AUC <= 0.55
# 12. Phase 6 recommendations caveated by actual model AUC


# Phase 2

## 1. Install dependencies

In [ ]:
!pip install pyspark mlflow --quiet


## 2. Imports (all phases consolidated)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from functools import reduce
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.spark
from mlflow.tracking import MlflowClient


## 3. Initialize Spark session (single session for all phases)

In [ ]:
spark = SparkSession.builder \
    .appName("CampaignAnalysis") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.ansi.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)


## 4. URLs + Helper functions (defined once, used across all phases)

In [ ]:
# GitHub raw URLs for CSV files
urls = {
    "nykaa":   "data/nykaa_campaign_data.csv",
    "purplle": "data/purplle_campaign_data.csv",
    "tira":    "data/tira_campaign_data.csv"
}

# Silver and Gold parquet save paths
silver_path = "data/marketing_silver_parquet"
gold_path   = "data/marketing_gold_parquet"

def normalize_colname(c: str) -> str:
    c = c.strip().lower()
    c = re.sub(r"[^a-z0-9]+", "_", c)
    c = re.sub(r"_+", "_", c).strip("_")
    return c

def parse_dates(pdf, col="date"):
    def try_parse(val):
        if pd.isnull(val):
            return None
        s = str(val).strip()
        try:
            return pd.to_datetime(s, format="%d-%m-%Y")
        except:
            pass
        try:
            return pd.to_datetime(s, format="%m/%d/%Y")
        except:
            pass
        return None
    pdf[col] = pdf[col].apply(try_parse)
    return pdf

def read_campaign(url, brand):
    pdf = pd.read_csv(url)
    pdf.columns = [normalize_colname(c) for c in pdf.columns]
    pdf = parse_dates(pdf, "date")
    pdf["brand_source"] = brand
    return spark.createDataFrame(pdf)

def align_cols(df, all_cols):
    for c in all_cols:
        if c not in df.columns:
            df = df.withColumn(c, F.lit(None))
    return df.select(all_cols)

def safe_div(num_col, den_col):
    return F.when(
        (den_col.isNotNull()) & (den_col > 0) & (num_col.isNotNull()),
        num_col / den_col
    )

print("Helper functions defined.")


## 5. Ingest and union all sources

In [ ]:
dfs = [read_campaign(url, brand) for brand, url in urls.items()]
all_cols = sorted(set().union(*[set(d.columns) for d in dfs]))
dfs_aligned = [align_cols(d, all_cols) for d in dfs]
df_all = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs_aligned)
df_all = df_all.dropDuplicates()

df_all.printSchema()
total_rows = df_all.count()
print(f"Total rows: {total_rows:,}")


## 6. Basic cleaning: nulls, whitespace, negative values

In [ ]:
string_cols = [c for c, t in df_all.dtypes if t == "string"]
df_all = df_all.fillna({c: "Unknown" for c in string_cols})
for c in string_cols:
    df_all = df_all.withColumn(c, F.trim(F.col(c)))

nonneg_cols = [c for c, t in df_all.dtypes
               if t in ("int", "bigint", "double", "float", "long")
               and c not in ("roi",)]
for c in nonneg_cols:
    df_all = df_all.withColumn(c, F.when(F.col(c) < 0, None).otherwise(F.col(c)))

print("Cleaning done.")


## 7. Feature engineering: KPI and funnel metrics

In [ ]:
df_feat = (df_all
    .withColumn("ctr",          safe_div(F.col("clicks"),           F.col("impressions")))
    .withColumn("lead_rate",    safe_div(F.col("leads"),            F.col("clicks")))
    .withColumn("lead_to_conv", safe_div(F.col("conversions"),      F.col("leads")))
    .withColumn("click_to_conv",safe_div(F.col("conversions"),      F.col("clicks")))
    .withColumn("cpc",          safe_div(F.col("acquisition_cost"), F.col("clicks")))
    .withColumn("cpa",          safe_div(F.col("acquisition_cost"), F.col("conversions")))
    .withColumn("roas",         safe_div(F.col("revenue"),          F.col("acquisition_cost")))
    .withColumn("rpm",          safe_div(F.col("revenue").cast("double") * 1000, F.col("impressions")))
)

for col in ["ctr", "lead_rate", "lead_to_conv", "click_to_conv"]:
    df_feat = df_feat.withColumn(
        col, F.when((F.col(col) < 0) | (F.col(col) > 1), None).otherwise(F.col(col))
    )

df_feat.select(
    "brand_source", "impressions", "clicks", "leads", "conversions",
    "ctr", "lead_rate", "lead_to_conv", "click_to_conv", "cpc", "cpa", "roas"
).show(10, truncate=False)


## 8. Save Silver layer

In [ ]:
df_feat.write.mode("overwrite").parquet(silver_path)
df_silver = spark.read.parquet(silver_path)
df_silver.createOrReplaceTempView("marketing_silver")
df_silver.printSchema()
print(f"Silver saved: {total_rows:,} rows")


# Phase 3

## 1. Summary statistics

In [ ]:
df_silver.summary().show(truncate=False)

spark.sql("""
SELECT
    COUNT(*)                    AS total_rows,
    ROUND(AVG(clicks), 2)       AS avg_clicks,
    ROUND(AVG(impressions), 2)  AS avg_impressions,
    ROUND(AVG(conversions), 2)  AS avg_conversions,
    ROUND(AVG(ctr), 4)          AS avg_ctr,
    ROUND(AVG(roi), 4)          AS avg_roi
FROM marketing_silver
""").show()


## 2. Feature distributions

In [ ]:
# Use a 10% sample before collecting to driver to keep memory usage reasonable
pdf = (df_silver
    .select("ctr", "click_to_conv", "cpa", "cpc",
            "acquisition_cost", "lead_rate", "roi", "engagement_score")
    .sample(fraction=0.1, seed=42)
    .dropna()
    .toPandas()
)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
features = ["ctr", "click_to_conv", "cpa", "cpc",
            "acquisition_cost", "lead_rate", "roi", "engagement_score"]
for ax, col in zip(axes.flat, features):
    sns.histplot(pdf[col], kde=True, ax=ax, bins=40, color="steelblue")
    ax.set_title(f"Distribution of {col}")

plt.suptitle("Feature Distributions (10% sample)", fontsize=14)
plt.tight_layout()
plt.show()


## 3. Correlation heatmap

In [ ]:
# Sample 10% before collecting; for larger datasets use pyspark.ml.stat.Correlation
corr_cols = ["impressions", "clicks", "conversions", "acquisition_cost",
             "ctr", "lead_rate", "click_to_conv", "lead_to_conv",
             "cpc", "cpa", "roi", "roas", "engagement_score"]

corr_matrix = (df_silver
    .select(corr_cols)
    .sample(fraction=0.1, seed=42)
    .dropna()
    .toPandas()
    .corr()
)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Feature Correlation Matrix (10% sample)")
plt.tight_layout()
plt.show()


## 4. Monthly ROI trend

In [ ]:
spark.sql("""
SELECT brand_source, MIN(date) AS start_date, MAX(date) AS end_date, COUNT(*) AS n_campaigns
FROM marketing_silver
GROUP BY brand_source
ORDER BY brand_source
""").show()

monthly_trend = spark.sql("""
SELECT
    DATE_FORMAT(date, 'yyyy-MM') AS month,
    ROUND(AVG(roi), 2)           AS avg_roi,
    ROUND(AVG(ctr), 4)           AS avg_ctr,
    COUNT(*)                     AS n_campaigns
FROM marketing_silver
GROUP BY DATE_FORMAT(date, 'yyyy-MM')
ORDER BY month
""").toPandas()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(monthly_trend["month"], monthly_trend["avg_roi"], marker="o", color="steelblue")
ax1.set_xlabel("Month")
ax1.set_ylabel("Avg ROI", color="steelblue")
ax1.tick_params(axis="x", rotation=45)
ax2 = ax1.twinx()
ax2.bar(monthly_trend["month"], monthly_trend["n_campaigns"], alpha=0.3, color="gray")
ax2.set_ylabel("# Campaigns", color="gray")
plt.title("Monthly ROI Trend & Campaign Volume")
fig.tight_layout()
plt.show()


## 5. Performance by campaign type

In [ ]:
spark.sql("""
SELECT
    campaign_type,
    COUNT(*)                     AS n_campaigns,
    ROUND(AVG(roi), 2)           AS avg_roi,
    ROUND(AVG(ctr), 4)           AS avg_ctr,
    ROUND(AVG(click_to_conv), 4) AS avg_conversion,
    ROUND(AVG(cpa), 2)           AS avg_cpa
FROM marketing_silver
GROUP BY campaign_type
ORDER BY avg_roi DESC
""").show(truncate=False)


## 6. Channel performance analysis

In [ ]:
df_channel = (df_silver
    .withColumn("channel", F.explode(F.split(F.col("channel_used"), ",")))
    .withColumn("channel", F.trim(F.col("channel")))
)
df_channel.groupBy("channel") \
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("roi"), 4).alias("avg_roi"),
        F.round(F.avg("ctr"), 4).alias("avg_ctr")
    ) \
    .orderBy(F.desc("avg_roi")) \
    .show(truncate=False)


## 7. Save Gold layer (EDA reference label only)

> **Note:** This gold label uses a threshold computed from the full dataset
> for EDA reference purposes only. Phase 4 recomputes the threshold exclusively
> on the training split to prevent label leakage into the test set.


In [ ]:
q75_ref = df_silver.approxQuantile("roi", [0.75], 0.01)[0]
print(f"75th percentile ROI (full dataset, EDA reference): {q75_ref}")

df_gold_eda = df_silver.withColumn(
    "label", F.when(F.col("roi") >= q75_ref, 1).otherwise(0)
)
df_gold_eda.write.mode("overwrite").parquet(gold_path)
df_gold_eda = spark.read.parquet(gold_path)
df_gold_eda.createOrReplaceTempView("marketing_gold")

df_gold_eda.groupBy("label").count().orderBy("label").show()
print(f"Gold (EDA) saved: {total_rows:,} rows")


# Phase 4

## 1. Load from Silver parquet

Phase 4 reads from the already-saved Silver parquet rather than re-running
the full ETL pipeline. This is what the medallion architecture is designed for.


In [ ]:
# Read from saved Silver parquet (ETL already done in Phase 2/3)
df_silver_p4 = spark.read.parquet(silver_path)

# Add pre-campaign date features
df_model = (df_silver_p4
    .withColumn("year",      F.year("date"))
    .withColumn("month",     F.month("date"))
    .withColumn("dayofweek", F.dayofweek("date"))
)

# Create binary channel flag columns from multi-value channel_used
df_model = df_model.withColumn("channel_used_lc", F.lower(F.col("channel_used")))
for ch in ["facebook", "whatsapp", "youtube", "google", "email", "instagram"]:
    df_model = df_model.withColumn(
        f"ch_{ch}",
        F.when(F.col("channel_used_lc").contains(ch), 1).otherwise(0)
    )
df_model = df_model.drop("channel_used_lc")

for c in ["duration", "year", "month", "dayofweek"]:
    df_model = df_model.withColumn(c, F.col(c).cast("double"))

print(f"Loaded {total_rows:,} rows from Silver parquet")


## 2. Define feature columns

> `target_audience` is a pre-campaign attribute available before a campaign
> launches and is now included as a categorical feature.


In [ ]:
categorical_cols = ["brand_source", "campaign_type", "customer_segment",
                    "language", "target_audience"]
numeric_cols     = ["duration", "year", "month", "dayofweek"]
channel_cols     = ["ch_facebook", "ch_whatsapp", "ch_youtube",
                    "ch_google", "ch_email", "ch_instagram"]

print("Categorical:", categorical_cols)
print("Numeric:",     numeric_cols)
print("Channels:",    channel_cols)


## 3. Train/test split (before label creation)

> Splitting **before** computing the ROI threshold ensures the threshold is
> derived exclusively from training data, preventing label leakage into the test set.


In [ ]:
train_raw, test_raw = df_model.randomSplit([0.8, 0.2], seed=42)
train_raw.cache()
test_raw.cache()
print(f"Raw split — Train: {train_raw.count():,}  |  Test: {test_raw.count():,}")


## 4. Create Gold label using training-set threshold only

In [ ]:
# Compute q75 on training data only (relativeError=0.0 for exact result)
q75 = train_raw.approxQuantile("roi", [0.75], 0.0)[0]
print(f"75th percentile ROI (train-only threshold): {q75:.4f}")

train_df = train_raw.withColumn("label", F.when(F.col("roi") >= q75, 1).otherwise(0))
test_df  = test_raw.withColumn( "label", F.when(F.col("roi") >= q75, 1).otherwise(0))

train_df.cache()
test_df.cache()
train_raw.unpersist()
test_raw.unpersist()

print("Label distribution (train):")
train_df.groupBy("label").count().orderBy("label").show()
print("Label distribution (test):")
test_df.groupBy("label").count().orderBy("label").show()


## 5. Build Spark ML pipeline stages

In [ ]:
indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in categorical_cols
]
encoder = OneHotEncoder(
    inputCols=[c + "_idx" for c in categorical_cols],
    outputCols=[c + "_ohe" for c in categorical_cols]
)
ohe_cols = [c + "_ohe" for c in categorical_cols]

assembler = VectorAssembler(
    inputCols=numeric_cols + channel_cols + ohe_cols,
    outputCol="features",
    handleInvalid="keep"
)
print("Pipeline stages defined.")


## 6. Evaluator helper function

In [ ]:
def evaluate_model(pred_df, model_name=""):
    auc_eval  = BinaryClassificationEvaluator(
        labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
    prec_eval = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
    rec_eval  = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="weightedRecall")
    f1_eval   = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="f1")
    metrics = {
        "AUC":       round(auc_eval.evaluate(pred_df),  4),
        "Precision": round(prec_eval.evaluate(pred_df), 4),
        "Recall":    round(rec_eval.evaluate(pred_df),  4),
        "F1":        round(f1_eval.evaluate(pred_df),   4),
    }
    if model_name:
        print(f"\n{model_name} Results:")
        for k, v in metrics.items():
            print(f"  {k}: {v}")
    return metrics


## 7. Train Logistic Regression with CrossValidator

In [ ]:
lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=50)
pipeline_lr = Pipeline(stages=indexers + [encoder, assembler, lr])
paramGrid_lr = (ParamGridBuilder()
    .addGrid(lr.regParam,        [0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0,  0.5])
    .build())
cv_lr = CrossValidator(
    estimator=pipeline_lr, estimatorParamMaps=paramGrid_lr,
    evaluator=BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC"),
    numFolds=3, seed=42)

print("Training Logistic Regression...")
cv_model_lr = cv_lr.fit(train_df)
pred_lr     = cv_model_lr.bestModel.transform(test_df)
metrics_lr  = evaluate_model(pred_lr, "Logistic Regression")


## 8. Train Random Forest with CrossValidator

In [ ]:
rf = RandomForestClassifier(labelCol="label", featuresCol="features")
pipeline_rf = Pipeline(stages=indexers + [encoder, assembler, rf])
paramGrid_rf = (ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100])
    .addGrid(rf.maxDepth, [3,   5])
    .build())
cv_rf = CrossValidator(
    estimator=pipeline_rf, estimatorParamMaps=paramGrid_rf,
    evaluator=BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC"),
    numFolds=3, seed=42)

print("Training Random Forest...")
cv_model_rf = cv_rf.fit(train_df)
pred_rf     = cv_model_rf.bestModel.transform(test_df)
metrics_rf  = evaluate_model(pred_rf, "Random Forest")


## 9. Train GBT with CrossValidator

In [ ]:
gbt = GBTClassifier(labelCol="label", featuresCol="features", maxIter=20)
pipeline_gbt = Pipeline(stages=indexers + [encoder, assembler, gbt])
paramGrid_gbt = (ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3,    5])
    .addGrid(gbt.stepSize, [0.05, 0.1])
    .build())
cv_gbt = CrossValidator(
    estimator=pipeline_gbt, estimatorParamMaps=paramGrid_gbt,
    evaluator=BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC"),
    numFolds=3, seed=42)

print("Training GBT...")
cv_model_gbt = cv_gbt.fit(train_df)
pred_gbt     = cv_model_gbt.bestModel.transform(test_df)
metrics_gbt  = evaluate_model(pred_gbt, "GBT Classifier")


## 10. Model comparison

In [ ]:
comparison = pd.DataFrame({
    "Model":     ["Logistic Regression", "Random Forest", "GBT Classifier"],
    "AUC":       [metrics_lr["AUC"],       metrics_rf["AUC"],       metrics_gbt["AUC"]],
    "Precision": [metrics_lr["Precision"], metrics_rf["Precision"], metrics_gbt["Precision"]],
    "Recall":    [metrics_lr["Recall"],    metrics_rf["Recall"],    metrics_gbt["Recall"]],
    "F1":        [metrics_lr["F1"],        metrics_rf["F1"],        metrics_gbt["F1"]],
})
print("=" * 60)
print("              Model Comparison")
print("=" * 60)
print(comparison.to_string(index=False))
print("=" * 60)

best_model_name = comparison.loc[comparison["AUC"].idxmax(), "Model"]
best_auc        = comparison["AUC"].max()
print(f"\nBest model: {best_model_name} (AUC: {best_auc:.4f})")

if best_auc <= 0.55:
    print("\n[WARNING] Best AUC <= 0.55: all models are performing near random chance.")
    print("          Pre-campaign features alone do not predict top-25% ROI.")
    print("          Feature importances from these models are not actionable.")

all_identical = (comparison["Recall"].nunique() == 1 and
                 comparison["Precision"].nunique() == 1)
if all_identical:
    print("\n[WARNING] All three models have identical Precision/Recall/F1.")
    print("          This indicates majority-class collapse: every row is predicted class 0.")


## 11. Feature importance

In [ ]:
best_model_name_map = {
    "Logistic Regression": (cv_model_lr, pred_lr),
    "Random Forest":       (cv_model_rf, pred_rf),
    "GBT Classifier":      (cv_model_gbt, pred_gbt),
}
best_cv_model, best_pred = best_model_name_map[best_model_name]
best_stage = best_cv_model.bestModel.stages[-1]

# Extract feature names from the best model's prediction df (not hardcoded to pred_gbt)
attrs = best_pred.schema["features"].metadata["ml_attr"]["attrs"]
features_list = []
for key in attrs:
    for a in attrs[key]:
        features_list.append(a)
features_list.sort(key=lambda x: x["idx"])
feature_names = [x["name"] for x in features_list]

if best_model_name == "Logistic Regression":
    coef = best_stage.coefficients.toArray()
    importance_df = pd.DataFrame({
        "Feature": feature_names, "Importance": abs(coef)
    }).sort_values("Importance", ascending=False).reset_index(drop=True)
    importance_label = "Coefficient Magnitude"
else:
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": best_stage.featureImportances.toArray()
    }).sort_values("Importance", ascending=False).reset_index(drop=True)
    importance_label = "Importance"

print(f"Top 10 Feature Importances ({best_model_name}):")
print(importance_df.head(10).to_string(index=False))

plt.figure(figsize=(10, 6))
plt.barh(importance_df["Feature"].head(10)[::-1],
         importance_df["Importance"].head(10)[::-1], color="steelblue")
plt.xlabel(importance_label)
plt.title(f"Top 10 Feature Importances ({best_model_name})")
plt.tight_layout()
plt.show()


## 12. Cleanup Phase 4 caches

In [ ]:
train_df.unpersist()
test_df.unpersist()
print("Phase 4 complete.")


# Phase 5

Phase 5 wraps the **already-trained** Phase 4 models with MLflow logging.
No re-training is performed. `cv_model_lr`, `cv_model_rf`, `cv_model_gbt`,
`pred_lr`, `pred_rf`, `pred_gbt`, and all metrics variables remain in scope
from Phase 4 and are reused directly.


## 1. MLflow experiment setup

In [ ]:
mlflow.set_experiment("Campaign_ROI_Classification")
print("MLflow experiment set: Campaign_ROI_Classification")


## 2. Log Logistic Regression run

In [ ]:
best_lr = cv_model_lr.bestModel.stages[-1]
with mlflow.start_run(run_name="LogisticRegression_CV"):
    mlflow.log_param("model",           "LogisticRegression")
    mlflow.log_param("regParam",        best_lr._java_obj.getRegParam())
    mlflow.log_param("elasticNetParam", best_lr._java_obj.getElasticNetParam())
    for k, v in metrics_lr.items():
        mlflow.log_metric(k, v)
    mlflow.spark.log_model(cv_model_lr.bestModel, artifact_path="spark_model")
    print(f"LR logged -> {metrics_lr}")


## 3. Log Random Forest run

In [ ]:
best_rf = cv_model_rf.bestModel.stages[-1]
with mlflow.start_run(run_name="RandomForest_CV"):
    mlflow.log_param("model",    "RandomForest")
    mlflow.log_param("numTrees", best_rf._java_obj.getNumTrees())
    mlflow.log_param("maxDepth", best_rf._java_obj.getMaxDepth())
    for k, v in metrics_rf.items():
        mlflow.log_metric(k, v)
    mlflow.spark.log_model(cv_model_rf.bestModel, artifact_path="spark_model")
    print(f"RF logged -> {metrics_rf}")


## 4. Log GBT run

In [ ]:
best_gbt = cv_model_gbt.bestModel.stages[-1]
with mlflow.start_run(run_name="GBT_CV"):
    mlflow.log_param("model",    "GBTClassifier")
    mlflow.log_param("maxDepth", best_gbt._java_obj.getMaxDepth())
    mlflow.log_param("stepSize", best_gbt._java_obj.getStepSize())
    for k, v in metrics_gbt.items():
        mlflow.log_metric(k, v)
    mlflow.spark.log_model(cv_model_gbt.bestModel, artifact_path="spark_model")
    print(f"GBT logged -> {metrics_gbt}")


## 5. Model comparison summary + log to MLflow

In [ ]:
print("=" * 60)
print("              Model Comparison (Phase 5)")
print("=" * 60)
print(comparison.to_string(index=False))
print("=" * 60)
print(f"Best model: {best_model_name} (AUC: {best_auc:.4f})")

with mlflow.start_run(run_name="Model_Comparison_Summary"):
    mlflow.log_metric("LR_AUC",  metrics_lr["AUC"])
    mlflow.log_metric("RF_AUC",  metrics_rf["AUC"])
    mlflow.log_metric("GBT_AUC", metrics_gbt["AUC"])
    mlflow.log_param("best_model", best_model_name)


## 6. Feature importance

In [ ]:
print(f"Top 10 Feature Importances ({best_model_name}):")
print(importance_df.head(10).to_string(index=False))

plt.figure(figsize=(10, 6))
plt.barh(importance_df["Feature"].head(10)[::-1],
         importance_df["Importance"].head(10)[::-1], color="steelblue")
plt.xlabel(importance_label)
plt.title(f"Top 10 Feature Importances ({best_model_name}) - Phase 5")
plt.tight_layout()
plt.show()


## 7. View MLflow runs

In [ ]:
runs = mlflow.search_runs(experiment_names=["Campaign_ROI_Classification"])
print(runs[["run_id", "tags.mlflow.runName",
            "metrics.AUC", "metrics.F1", "params.model"]].to_string(index=False))


## 8. Register best model + load and re-score

In [ ]:
client = MlflowClient()

runs_sorted = mlflow.search_runs(
    experiment_names=["Campaign_ROI_Classification"],
    order_by=["metrics.AUC DESC"]
)
model_runs = runs_sorted[
    runs_sorted["tags.mlflow.log-model.history"].str.contains("spark_model", na=False)
]
best_run = model_runs.iloc[0]

print(f"Best run ID  : {best_run['run_id']}")
print(f"Run name     : {best_run['tags.mlflow.runName']}")
print(f"Model        : {best_run['params.model']}")
print(f"AUC          : {best_run['metrics.AUC']:.4f}")
print(f"F1           : {best_run['metrics.F1']:.4f}")

model_uri    = f"runs:/{best_run['run_id']}/spark_model"
loaded_model = mlflow.spark.load_model(model_uri)
print(f"\nLoaded model from: {model_uri}")

pred_loaded    = loaded_model.transform(test_df)
metrics_loaded = evaluate_model(pred_loaded)
print(f"Loaded model metrics: {metrics_loaded}")


## 9. Cleanup

In [ ]:
train_df.unpersist()
test_df.unpersist()
print("Phase 5 complete.")


# Phase 6: Insights & Business Recommendations

In [ ]:
print("--- Key Findings ---")
print(f"Best Model      : {best_model_name} (AUC: {best_auc:.4f})")
print(f"ROI Threshold   : top 25% (>= {q75:.2f}, computed on training split only)")
print(f"Total Campaigns : {total_rows:,}")
print(f"\nModel Comparison:")
print(comparison.to_string(index=False))
print(f"\nTop 3 Features driving high ROI:")
for i, row in importance_df.head(3).iterrows():
    print(f"  {i+1}. {row['Feature']}: {row['Importance']:.4f}")

if best_auc <= 0.55:
    print("\n" + "!" * 65)
    print(f"! CAUTION: Best AUC = {best_auc:.4f} (near random chance = 0.50)")
    print("! Recommendations below are exploratory hypotheses only.")
    print("! Do NOT act on them without improved model performance (AUC > 0.65).")
    print("!" * 65)


## Business Recommendations

> **Model Performance Caveat:** The current best AUC is ~0.50-0.51, statistically
> indistinguishable from random chance. All recommendations below are **exploratory
> hypotheses** derived from feature importance and EDA. They must NOT be used to
> drive budget or campaign decisions until model AUC meaningfully improves (target > 0.65).
> Possible paths to improvement: adding external signals (seasonality, competitor spend,
> historical brand performance), engineering interaction features, or switching to
> regression (predict ROI directly instead of binary classification).

**1. Prioritize high-signal campaign attributes (pending model improvement)**
The feature importance analysis identified the top drivers of high-ROI campaigns.
Marketing teams should investigate whether these align with domain expertise.
If they do, those attributes are worth prioritizing once model quality is confirmed.

**2. Use the model for campaign pre-screening (only when AUC > 0.65)**
Before launching a campaign, input its planned attributes into the trained model
to predict whether it is likely high-performing. This reduces spend on low-ROI
campaigns. This step is only actionable after retraining with richer features.

**3. Optimize campaign timing**
Month and day-of-week features appeared in the model. Independently of model quality,
the Phase 3 monthly ROI trend chart provides a reliable signal for historical
high-ROI periods that can inform scheduling decisions.

**4. Challenges & Next Steps**
- Current AUC ~0.50 is a fundamental signal limitation: pre-campaign features alone
  are insufficient to predict top-25% ROI outcomes
- Adding external signals (seasonality index, brand awareness score, competitor
  spend estimates) could provide the missing discriminatory power
- Label threshold (q75) should be recalibrated on new data each training cycle
- Model should be retrained monthly as new campaign data arrives
- Consider a regression approach (predict ROI directly) rather than binary classification
